# Scrollie — Generic Segmentation Viewer

Interactive slice-by-slice viewer for any algorithm in `eval_notebooks/`.

**How to use:**
1. Run all cells.
2. Pick up to **two algorithms** from the dropdowns — the raw image is always shown on the left.
3. Pick a **stack** (updates automatically when you change Algorithm 1).
4. Drag the **slice slider** to scroll.

To add a new algorithm, add an entry to `ALGORITHMS` in the Configuration cell.

In [ ]:
import glob
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, VBox, HBox
from IPython.display import display

In [ ]:
# ── Pre-defined label maps for NIfTI algorithms ───────────────────────────────
# (NPZ algorithms detect their labels automatically from the array keys)

MM_WB_LABELS = {
    7101: 'Vastus_Lateralis_L',   7102: 'Vastus_Lateralis_R',
    7111: 'Vastus_Intermedius_L', 7112: 'Vastus_Intermedius_R',
    7121: 'Vastus_Medialis_L',    7122: 'Vastus_Medialis_R',
    7131: 'Rectus_Femoris_L',     7132: 'Rectus_Femoris_R',
    7141: 'Sartorius_L',          7142: 'Sartorius_R',
    7151: 'Gracilis_L',           7152: 'Gracilis_R',
    7161: 'Semimembranosus_L',    7162: 'Semimembranosus_R',
    7171: 'Semitendinosus_L',     7172: 'Semitendinosus_R',
    7181: 'Biceps_Femoris_L',     7182: 'Biceps_Femoris_R',
    7201: 'Adductor_Magnus_L',    7202: 'Adductor_Magnus_R',
}

MM_THIGH_LABELS = {
    9:  'L_sartorius',
    10: 'R_sartorius',
    11: 'L_gracilis',
    12: 'R_gracilis',
}

MUSEG_LABELS = {
    1:  'Vastus_Lateralis',
    2:  'Vastus_Intermedius',
    3:  'Vastus_Medialis',
    4:  'Rectus_Femoris',
    5:  'Sartorius',
    6:  'Gracilis',
    7:  'Semimembranosus',
    8:  'Semitendinosus',
    9:  'Biceps_Femoris',
    10: 'Biceps_Femoris_Short',
    11: 'Adductor_Magnus',
    12: 'Adductor_Longus',
    13: 'Adductor_Brevis',
}

HIRRIRIRIIR_LABELS = {
    1:  'Sartorius',
    2:  'Rectus_Femoris',
    3:  'Vastus_Lateralis',
    4:  'Vastus_Intermedius',
    5:  'Vastus_Medialis',
    6:  'Adductor_Magnus',
    7:  'Gracilis',
    8:  'Biceps_Femoris_Long',
    9:  'Semitendinosus',
    10: 'Semimembranosus',
    11: 'Biceps_Femoris_Short',
}

print('Label maps defined.')

In [ ]:
# ── Configuration — edit / extend this cell ───────────────────────────────────
EVAL_DIR = r'C:\Projects\dissector\eval_notebooks'
GT_BASE  = os.path.join(EVAL_DIR, 'myosegmenTUM')

# Each entry:
#   seg_dir   : folder containing segmentation files
#   suffix    : filename suffix used to find files (e.g. '_dseg.nii.gz')
#   fmt       : 'nifti' (integer labels) or 'npz' (string keys)
#   label_map : {int: name} for nifti — used to colour labels
#               None for npz — keys are read directly from the file
#   modality  : 'WATER' or 'FATFRACTION' — determines which raw image to load

ALGORITHMS = {
    'MuscleMap WB (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_wb', 'segs_water'),
        'suffix':    '_dseg.nii.gz',
        'fmt':       'nifti',
        'label_map': MM_WB_LABELS,
        'modality':  'WATER',
    },
    'MuscleMap WB (fat fraction)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_wb', 'segmentations_fat_frac'),
        'suffix':    '_dseg.nii.gz',
        'fmt':       'nifti',
        'label_map': MM_WB_LABELS,
        'modality':  'FATFRACTION',
    },
    'MM WB + MedSAM bbox (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_wb_boxes_medsam', 'segs_water'),
        'suffix':    '_mm_medsam.npz',
        'fmt':       'npz',
        'label_map': None,
        'modality':  'WATER',
    },
    'MM WB + MedSAM bbox (fat fraction)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_wb_boxes_medsam', 'segmentation_fat_fraction'),
        'suffix':    '_mm_medsam.npz',
        'fmt':       'npz',
        'label_map': None,
        'modality':  'FATFRACTION',
    },
    'MM WB + MedSAM logitmask (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_wb_masks_medsam', 'segs_water'),
        'suffix':    '_mm_logitmask.npz',
        'fmt':       'npz',
        'label_map': None,
        'modality':  'WATER',
    },
    'MM WB + MedSAM logitmask (fat fraction)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_wb_masks_medsam', 'segmentations_fat_fraction'),
        'suffix':    '_mm_logitmask.npz',
        'fmt':       'npz',
        'label_map': None,
        'modality':  'FATFRACTION',
    },
    'MM WB + SLM-SAM2 (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_wb+slmsam', 'segs_water'),
        'suffix':    '_slmsam2.npz',
        'fmt':       'npz',
        'label_map': None,
        'modality':  'WATER',
    },
    'MM WB + SLM-SAM2 (fat fraction)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_wb+slmsam', 'segmentation_fat_fraction'),
        'suffix':    '_slmsam2.npz',
        'fmt':       'npz',
        'label_map': None,
        'modality':  'FATFRACTION',
    },
    'MedCLIP-SAMv2 (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'medclipsamv2', 'segs_water'),
        'suffix':    '_medclipsamv2.npz',
        'fmt':       'npz',
        'label_map': None,
        'modality':  'WATER',
    },
    'MedCLIP-SAMv2 (fat fraction)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'medclipsamv2', 'segmentations_fat_frac'),
        'suffix':    '_medclipsamv2.npz',
        'fmt':       'npz',
        'label_map': None,
        'modality':  'FATFRACTION',
    },
    'Hirriririir / Multimodal Thigh (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'multimodal-multiethnic', 'segs_water'),
        'suffix':    '_thigh_seg.nii.gz',
        'fmt':       'nifti',
        'label_map': HIRRIRIRIIR_LABELS,
        'modality':  'WATER',
    },
    'Hirriririir / Multimodal Thigh (fat fraction)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'multimodal-multiethnic', 'segmentations_fat_frac'),
        'suffix':    '_thigh_seg.nii.gz',
        'fmt':       'nifti',
        'label_map': HIRRIRIRIIR_LABELS,
        'modality':  'FATFRACTION',
    },
    'Dafne (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'dafne', 'segs_water'),
        'suffix':    '_dafne_thigh.npz',
        'fmt':       'npz',
        'label_map': None,
        'modality':  'WATER',
    },
    'Dafne + MedSAM (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'dafne_and_medsam', 'segs_water'),
        'suffix':    '_dafne_medsam.npz',
        'fmt':       'npz',
        'label_map': None,
        'modality':  'WATER',
    },
    'MuscleMap Thigh (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_thigh', 'segs_water'),
        'suffix':    '_dseg.nii.gz',
        'fmt':       'nifti',
        'label_map': MM_THIGH_LABELS,
        'modality':  'WATER',
    },
    'MuSeg Thigh (water)': {
        'seg_dir':   os.path.join(EVAL_DIR, 'museg', 'segs_water'),
        'suffix':    '_museg.nii.gz',
        'fmt':       'nifti',
        'label_map': MUSEG_LABELS,
        'modality':  'WATER',
    },
}

# Only show algorithms whose seg_dir actually exists
AVAILABLE = {name: cfg for name, cfg in ALGORITHMS.items()
             if os.path.isdir(cfg['seg_dir'])}
print(f'Available algorithms ({len(AVAILABLE)}/{len(ALGORITHMS)}):')
for name in AVAILABLE:
    n = len(glob.glob(os.path.join(AVAILABLE[name]['seg_dir'],
                                   f'*{AVAILABLE[name]["suffix"]}')))
    print(f'  {name}: {n} files')

In [ ]:
# ── Loading helpers ───────────────────────────────────────────────────────────

def parse_stem(filename, suffix):
    """Extract (subject, modality, stack_num) from a segmentation filename."""
    stem = os.path.basename(filename)
    if stem.endswith(suffix):
        stem = stem[: -len(suffix)]
    m = re.match(r'(.+?)_(WATER|FATFRACTION)_stack(\d+)', stem)
    if not m:
        return None, None, None
    return m.group(1), m.group(2), m.group(3)


def image_path(subject, modality, stack_num):
    """Return the raw NIfTI path for a given subject / modality / stack."""
    return os.path.join(
        GT_BASE, subject, 'ImageData',
        f'{subject}_{modality}',
        f'{subject}_{modality}_stack{stack_num}.nii',
    )


def load_image(subject, modality, stack_num):
    """Load and normalise the raw MRI stack to [0, 1]."""
    path = image_path(subject, modality, stack_num)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Raw image not found: {path}')
    arr = sitk.GetArrayFromImage(sitk.ReadImage(path)).astype(float)
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo + 1e-8)


def build_overlay(seg_seq, label_map, cmap):
    """Build (D, H, W, 4) RGBA overlay from a sequential integer label volume."""
    overlay = np.zeros((*seg_seq.shape, 4), dtype=float)
    for seq_idx, name in label_map.items():
        r, g, b, _ = cmap(seq_idx - 1)
        overlay[seg_seq == seq_idx] = [r, g, b, 0.5]
    return overlay


def load_seg(seg_path, cfg):
    """
    Load a segmentation file and return (overlay, label_map, cmap, legend_patches).
    Works for both NIfTI (integer labels) and NPZ (string keys).
    """
    if cfg['fmt'] == 'nifti':
        raw_label_map = cfg['label_map']
        seg_arr = sitk.GetArrayFromImage(sitk.ReadImage(seg_path))
        # Map original labels to sequential 1-N (only labels present in this file)
        present = {k: v for k, v in raw_label_map.items() if np.any(seg_arr == k)}
        seq_map   = {i: name for i, (_, name) in enumerate(present.items(), start=1)}
        orig_to_seq = {orig: i for i, (orig, _) in enumerate(present.items(), start=1)}
        seg_seq = np.zeros_like(seg_arr, dtype=np.uint16)
        for orig_k, seq_i in orig_to_seq.items():
            seg_seq[seg_arr == orig_k] = seq_i

    else:  # npz
        data    = np.load(seg_path)
        names   = list(data.files)
        seq_map = {i: name for i, name in enumerate(names, start=1)}
        shape   = data[names[0]].shape
        seg_seq = np.zeros(shape, dtype=np.uint16)
        for seq_i, name in seq_map.items():
            seg_seq[data[name] > 0] = seq_i

    n = max(len(seq_map), 1)
    cmap    = plt.colormaps['tab20'].resampled(n)
    overlay = build_overlay(seg_seq, seq_map, cmap)
    patches = [
        mpatches.Patch(color=cmap(i - 1), alpha=0.6, label=name)
        for i, name in seq_map.items()
    ]
    return overlay, seq_map, cmap, patches


def get_stacks(algo_name):
    """Return {display_label: seg_path} for all files found for an algorithm."""
    cfg = AVAILABLE[algo_name]
    files = sorted(glob.glob(os.path.join(cfg['seg_dir'], f'*{cfg["suffix"]}')))
    result = {}
    for f in files:
        subj, mod, stack = parse_stem(f, cfg['suffix'])
        if subj:
            label = f'{subj}_{mod}_stack{stack}'
            result[label] = f
    return result


print('Helpers ready.')

In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────────

ALGO_OPTIONS = ['— none —'] + list(AVAILABLE)

algo1_dd = Dropdown(options=list(AVAILABLE), description='Algorithm 1:',
                    layout=widgets.Layout(width='420px'))
algo2_dd = Dropdown(options=ALGO_OPTIONS, description='Algorithm 2:',
                    value='— none —',
                    layout=widgets.Layout(width='420px'))
stack_dd = Dropdown(options=[], description='Stack:',
                    layout=widgets.Layout(width='420px'))
slice_sl = IntSlider(min=0, max=1, step=1, value=0, description='Slice:',
                     layout=widgets.Layout(width='600px'))
out = widgets.Output()

_cache = {}


def _load(algo_name, stack_label):
    key = (algo_name, stack_label)
    if key not in _cache:
        cfg      = AVAILABLE[algo_name]
        stacks   = get_stacks(algo_name)
        seg_path = stacks[stack_label]
        subj, mod, stack_num = parse_stem(seg_path, cfg['suffix'])
        img_norm             = load_image(subj, cfg['modality'], stack_num)
        overlay, seq_map, cmap, patches = load_seg(seg_path, cfg)
        _cache[key] = (img_norm, overlay, patches)
    return _cache[key]


def render(algo1, algo2, stack_label, slice_idx):
    if not stack_label:
        return
    try:
        img_norm, ov1, patches1 = _load(algo1, stack_label)
    except Exception as e:
        with out:
            out.clear_output(wait=True)
            print(f'Error loading {algo1}: {e}')
        return

    has_algo2 = algo2 != '— none —' and algo2 in AVAILABLE
    ov2, patches2 = None, []
    if has_algo2:
        try:
            _, ov2, patches2 = _load(algo2, stack_label)
        except Exception as e:
            has_algo2 = False
            print(f'Could not load {algo2}: {e}')

    img = img_norm[slice_idx]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Panel 1 — raw image, no overlay
    axes[0].imshow(img, cmap='gray', origin='lower')
    axes[0].set_title('Raw image (unsegmented)', fontsize=10)
    axes[0].axis('off')

    # Panel 2 — Algorithm 1
    axes[1].imshow(img, cmap='gray', origin='lower')
    axes[1].imshow(ov1[slice_idx], origin='lower')
    axes[1].set_title(algo1, fontsize=10)
    axes[1].axis('off')
    axes[1].legend(handles=patches1, loc='lower right', fontsize=5,
                   framealpha=0.7, ncol=2)

    # Panel 3 — Algorithm 2 (or blank placeholder)
    axes[2].imshow(img, cmap='gray', origin='lower')
    if has_algo2:
        axes[2].imshow(ov2[slice_idx], origin='lower')
        axes[2].set_title(algo2, fontsize=10)
        axes[2].legend(handles=patches2, loc='lower right', fontsize=5,
                       framealpha=0.7, ncol=2)
    else:
        axes[2].set_title('Algorithm 2 — select above', fontsize=10, color='gray')
    axes[2].axis('off')

    fig.suptitle(f'{stack_label}  —  slice {slice_idx}', fontsize=11)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()


def on_algo1_change(change):
    _cache.clear()
    stacks = get_stacks(change['new'])
    stack_dd.options = list(stacks)
    if stacks:
        stack_dd.value = list(stacks)[0]
        slice_sl.max   = _load(change['new'], stack_dd.value)[0].shape[0] - 1
        slice_sl.value = 0
    render(algo1_dd.value, algo2_dd.value, stack_dd.value, slice_sl.value)


def on_algo2_change(change):
    render(algo1_dd.value, algo2_dd.value, stack_dd.value, slice_sl.value)


def on_stack_change(change):
    if change['new']:
        img_norm, _, _ = _load(algo1_dd.value, change['new'])
        slice_sl.max   = img_norm.shape[0] - 1
        slice_sl.value = 0
    render(algo1_dd.value, algo2_dd.value, change['new'], slice_sl.value)


def on_slice_change(change):
    render(algo1_dd.value, algo2_dd.value, stack_dd.value, change['new'])


algo1_dd.observe(on_algo1_change, names='value')
algo2_dd.observe(on_algo2_change, names='value')
stack_dd.observe(on_stack_change, names='value')
slice_sl.observe(on_slice_change, names='value')

# Initial load
if AVAILABLE:
    stacks = get_stacks(algo1_dd.value)
    stack_dd.options = list(stacks)
    if stacks:
        stack_dd.value = list(stacks)[0]
        img0, _, _ = _load(algo1_dd.value, stack_dd.value)
        slice_sl.max = img0.shape[0] - 1
    render(algo1_dd.value, algo2_dd.value, stack_dd.value, 0)

display(VBox([
    HBox([algo1_dd, algo2_dd]),
    HBox([stack_dd, slice_sl]),
    out,
]))